In [ ]:
# cell 1 — install
!pip install deepforest

# cell 2 — zero-shot detection on your ortho
from deepforest import main
import pandas as pd

m = main.deepforest()
m.load_model("/kaggle/working/deepforest_tcd_eu.pt")  # instead of weecology default
m.config["score_thresh"] = 0.1  

predictions = m.predict_tile(
    path="/kaggle/input/datasets/tercasaskova/bologna-tiff/bologna_rgb_2023.tif",
    patch_size=400,
    patch_overlap=0.10,
    iou_threshold=0.15,
    dataloader_strategy="window",  # memory safe — never loads full image
)

predictions.to_csv("/kaggle/working/raw_predictions_checkpoint.csv", index=False)
print(predictions.head())
print(f"{len(predictions):,} detections")

In [3]:
!pip install deepforest datasets torch torchvision --quiet
import os
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import json, os, shutil
from PIL import Image
import pandas as pd
from pathlib import Path
from deepforest import main
import torch
from sklearn.model_selection import train_test_split
from deepforest import main

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

from datasets import load_dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 82.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00


In [2]:
!pip install deepforest datasets torch torchvision --quiet
import os
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import json, os, shutil
from PIL import Image
import pandas as pd
from pathlib import Path
from deepforest import main
import torch
from sklearn.model_selection import train_test_split
from deepforest import main

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

from datasets import load_dataset

ds = load_dataset("restor/tcd", split="train", streaming=True)

# filter european on the fly
european = (
    sample for sample in ds
    if 35 <= sample["lat"] <= 72 and -25 <= sample["lon"] <= 45
)


TILES_DIR = Path("/kaggle/working/tcd_tiles")
TILES_DIR.mkdir(exist_ok=True)

# Rebuild — generator was exhausted after Cell 2
ds = load_dataset("restor/tcd", split="train", streaming=True)
european = (
    sample for sample in ds
    if 35 <= sample["lat"] <= 72 and -25 <= sample["lon"] <= 45
)

MAX_TILES = 2000  # safety cap; remove or raise when you're happy with the pipeline
rows = []

for i, sample in enumerate(european):
    if i >= MAX_TILES:
        break

    img_path = TILES_DIR / f"tile_{i:04d}.png"
    if not img_path.exists():          # skip re-saving on reruns
        sample["image"].save(img_path)

    raw = sample.get("coco_annotations")
    if not raw:
        continue
    try:
        annots = json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        continue

    # coco_annotations may be a list directly or wrapped in a dict
    if isinstance(annots, dict):
        annots = annots.get("annotations", [])

    for ann in annots:
        if ann.get("category_id") != 2:   # 2 = individual tree
            continue
        if ann.get("iscrowd", 0):
            continue
        x, y, w, h = ann["bbox"]
        if w < 5 or h < 5:
            continue
        rows.append({
            "image_path": str(img_path),
            "xmin": float(x),
            "ymin": float(y),
            "xmax": float(x + w),
            "ymax": float(y + h),
            "label": "Tree",
        })

df = pd.DataFrame(rows)
print(f"rows collected: {len(df)}")        # debug before assuming it worked
assert "image_path" in df.columns, "Still empty — check coco_annotations structure"

df.to_csv("/kaggle/working/tcd_european_train.csv", index=False)
print(f"✓ {len(df):,} boxes from {df['image_path'].nunique()} tiles")

IMG_SIZE = 2048

df["xmin"] = df["xmin"].clip(lower=0)
df["ymin"] = df["ymin"].clip(lower=0)
df["xmax"] = df["xmax"].clip(upper=IMG_SIZE)
df["ymax"] = df["ymax"].clip(upper=IMG_SIZE)

# drop boxes that became too small after clipping
df = df[((df["xmax"] - df["xmin"]) >= 5) & ((df["ymax"] - df["ymin"]) >= 5)]

model = main.deepforest()
model.load_model(model_name="weecology/deepforest-tree", revision="main")

# split first
tiles = df["image_path"].unique()
train_tiles, val_tiles = train_test_split(tiles, test_size=0.15, random_state=42)
df[df["image_path"].isin(train_tiles)].to_csv("/kaggle/working/tcd_european_train.csv", index=False)
df[df["image_path"].isin(val_tiles)].to_csv("/kaggle/working/tcd_european_val.csv", index=False)

# all config before create_trainer
model.config["train"]["csv_file"] = "/kaggle/working/tcd_european_train.csv"
model.config["train"]["root_dir"] = "/kaggle/working/tcd_tiles"
model.config["validation"]["csv_file"] = "/kaggle/working/tcd_european_val.csv"
model.config["validation"]["root_dir"] = "/kaggle/working/tcd_tiles"
model.config["train"]["epochs"] = 15
model.config["train"]["lr"] = 1e-4
model.config["train"]["batch_size"] = 4
model.config["train"]["augment"] = True
model.config["score_thresh"] = 0.3
model.config["nms_thresh"] = 0.15

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model.create_trainer(fast_dev_run=False)
model.trainer.fit(model)
model.save_model("/kaggle/working/deepforest_tcd_eu.pt")
print("Model saved.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 8.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00


README.md: 0.00B [00:00, ?B/s]

rows collected: 69626
✓ 69,626 boxes from 688 tiles


config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


...

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100
detections in a single image. This means that certain detections with the lowest scores will be ignored, that may 
have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use
case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100
detections in a single image. This means that certain detections with the lowest scores will be ignored, that may 
have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use
case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)

`Trainer.fit` stopped: `max_epochs=15` reached.


`weights_only` was not set, defaulting to `False`.


Model saved.


In [ ]:
from deepforest import main
import torch

m = main.deepforest()
m.load_model(model_name="weecology/deepforest-tree", revision="main")  # init architecture via HF

state = torch.load(
    "/kaggle/input/datasets/tercasaskova/training-finetuning/deepforest_tcd_eu.pt",
    map_location="cuda" if torch.cuda.is_available() else "cpu",
)
if "state_dict" in state:
    m.model.load_state_dict(state["state_dict"])
else:
    m.model.load_state_dict(state)

m.config["score_thresh"] = 0.1
print("✓ finetuned model loaded")

predictions = m.predict_tile(
    path="/kaggle/input/datasets/tercasaskova/bologna-tiff/bologna_rgb_2023.tif",
    patch_size=400,
    patch_overlap=0.10,
    iou_threshold=0.15,
    dataloader_strategy="window",
)

predictions.to_csv("/kaggle/working/raw_predictions_checkpoint.csv", index=False)
print(predictions.head())
print(f"{len(predictions):,} detections")

config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s] 


model.safetensors:   0%|          | 0.00/129M [00:00<?, ?B/s]

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


✓ finetuned model loaded


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()